In [1]:
#course catalog link: https://catalog.ucsd.edu/front/courses.html

#!pip install ace-tools







In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Base URL for UCSD catalog (change to the appropriate department)
BASE_URL = "https://catalog.ucsd.edu/courses/DSC.html"  # Example for Data Science courses

def scrape_ucsd_courses(url):
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Failed to fetch {url}")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    courses = []

    # Find all course names and descriptions
    course_names = soup.find_all("p", class_="course-name")
    course_descriptions = soup.find_all("p", class_="course-descriptions")

    for i in range(len(course_names)):
        try:
            # Extract Course Name
            course_name_text = course_names[i].text.strip()

            # Extract Course ID and Title separately
            if ". " in course_name_text:
                course_id, course_title = course_name_text.split(". ", 1)
            else:
                course_id, course_title = course_name_text, ""

            # Extract Course Description (check if exists)
            course_description = course_descriptions[i].text.strip() if i < len(course_descriptions) else "No description available"

            # Append to list
            courses.append({
                "Course ID": course_id,
                "Course Name": course_title,
                "Description": course_description,
            })

        except Exception as e:
            print(f"Error processing course: {e}")

    return courses

# Run scraper
course_data = scrape_ucsd_courses(BASE_URL)

# Convert to DataFrame
df = pd.DataFrame(course_data)
display(df)


,Course ID,Course Name,Description
0,DSC 10,Principles of Data Science (4),This first course in data science introduces s...
1,DSC 20,Programming and Basic Data Structures for Data...,Provides an understanding of the structures th...
2,DSC 30,Data Structures and Algorithms for Data Scienc...,Builds on topics covered in DSC 20 and provide...
3,DSC 40A,Theoretical Foundations of Data Science I (4),The sequence DSC 40A-B introduces the theoreti...
4,DSC 40B,Theoretical Foundations of Data Science II (4),"The marriage of data, computation, and inferen..."
...,...,...,...
75,DSC 295,Academia Survival Skills (1),Internship with institution or agency allowing...
76,DSC 297. DSC Graduate Internship (1),,Graduate research. May be taken for credit up ...
77,DSC 299,Graduate Research (1–16),A course in which teaching assistants are aide...
78,DSC 500,Teaching Assistantship (2 or 4),Training in teaching methods in the field of d...


In [1]:
PINECONE_API_KEY = "pcsk_5b67TL_BgErCnfngE1K8C6Kh4SuPuiFZNDyALbvtqi1fUn1gxzEK5p2KYm4nue6fwxEp4C"
index_name = "dscourses"

In [6]:
import pinecone
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
import pandas as pd
import json


pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(index_name)

model = SentenceTransformer("intfloat/multilingual-e5-large")
print("Model output dimension:", model.get_sentence_embedding_dimension())

index = pc.Index(index_name)

# Load the embedding model



def insert_courses_into_pinecone(df):
    vectors = []
    for i, row in df.iterrows():
        try:
            if row["Course ID"]:
                course_id = "".join(c if c.isalnum() else "" for c in row["Course ID"])
                #course_id = row["Course ID"]
            else: 
                course_id = "No Course ID"
            print(course_id)
            text = f"{row['Course Name']}: {row['Description']}"
            
            # Generate embedding
            embedding = model.encode(text).tolist()

            # Add to vector list
            course_name = row["Course Name"].strip() if row["Course Name"] else "No Course Name"


            metadata = {"Course Name": course_name, "Description": row["Description"]}
            vectors.append((course_id, embedding, metadata))

            print(f"✅ Course {i+1}/{len(df)} - {course_id} added to batch.")

        except Exception as e:
            print(f"❌ Error processing course {i+1}: {e}")

    # Insert into Pinecone in batches
    batch_size = 100
    for i in range(0, len(vectors), batch_size):
        batch = vectors[i : i + batch_size]
        index.upsert(vectors=batch)
        print(f"📦 Batch {i//batch_size + 1} ({len(batch)} courses) uploaded to Pinecone.")

    print("🚀 Data successfully inserted into Pinecone!")

# Run insertion
insert_courses_into_pinecone(df)


✅ Model loaded with dimension: 1024


Processing Courses:   0%|          | 0/7333 [00:00<?, ?it/s]

🔹 Processing Course 1/7333: AIP97
❌ Error processing course 1: 'float' object is not iterable
🔹 Processing Course 2/7333: AIP197
❌ Error processing course 2: 'float' object is not iterable
🔹 Processing Course 3/7333: AIP197DC
❌ Error processing course 3: 'float' object is not iterable
🔹 Processing Course 4/7333: AIP197P
❌ Error processing course 4: 'float' object is not iterable
🔹 Processing Course 5/7333: AIP197T


Processing Courses:   0%|          | 16/7333 [00:00<01:39, 73.34it/s]

✅ Course AIP197T added to batch.
🔹 Processing Course 6/7333: AAS10
✅ Course AAS10 added to batch.
🔹 Processing Course 7/7333: AAS11
❌ Error processing course 7: 'float' object is not iterable
🔹 Processing Course 8/7333: AAS14
❌ Error processing course 8: 'float' object is not iterable
🔹 Processing Course 9/7333: AAS15
❌ Error processing course 9: 'float' object is not iterable
🔹 Processing Course 10/7333: AAS87
❌ Error processing course 10: 'float' object is not iterable
🔹 Processing Course 11/7333: AAS170
✅ Course AAS170 added to batch.
🔹 Processing Course 12/7333: AAS171
❌ Error processing course 12: 'float' object is not iterable
🔹 Processing Course 13/7333: AAS172
❌ Error processing course 13: 'float' object is not iterable
🔹 Processing Course 14/7333: AAS179
❌ Error processing course 14: 'float' object is not iterable
🔹 Processing Course 15/7333: AASANSC185
❌ Error processing course 15: 'float' object is not iterable
🔹 Processing Course 16/7333: AAS190
✅ Course AAS190 added to bat

...

✅ Course AWP102 added to batch.
🔹 Processing Course 31/7333: AWP102R
❌ Error processing course 31: 'float' object is not iterable
🔹 Processing Course 32/7333: AWP103
❌ Error processing course 32: 'float' object is not iterable
🔹 Processing Course 33/7333: AWP104
❌ Error processing course 33: 'float' object is not iterable
🔹 Processing Course 34/7333: ANTH1
✅ Course ANTH1 added to batch.
🔹 Processing Course 35/7333: ANTH2
✅ Course ANTH2 added to batch.
🔹 Processing Course 36/7333: ANTH3


Processing Courses:   0%|          | 36/7333 [00:01<09:50, 12.36it/s]

✅ Course ANTH3 added to batch.
🔹 Processing Course 37/7333: ANTH4
✅ Course ANTH4 added to batch.
🔹 Processing Course 38/7333: ANTH5


Processing Courses:   1%|          | 37/7333 [00:02<07:44, 15.70it/s]


KeyboardInterrupt: 

In [13]:
df

,Course ID,Course Name,Description
0,DSC 10,Principles of Data Science (4),This first course in data science introduces s...
1,DSC 20,Programming and Basic Data Structures for Data...,Provides an understanding of the structures th...
2,DSC 30,Data Structures and Algorithms for Data Scienc...,Builds on topics covered in DSC 20 and provide...
3,DSC 40A,Theoretical Foundations of Data Science I (4),The sequence DSC 40A-B introduces the theoreti...
4,DSC 40B,Theoretical Foundations of Data Science II (4),"The marriage of data, computation, and inferen..."
...,...,...,...
75,DSC 295,Academia Survival Skills (1),Internship with institution or agency allowing...
76,DSC 297. DSC Graduate Internship (1),,Graduate research. May be taken for credit up ...
77,DSC 299,Graduate Research (1–16),A course in which teaching assistants are aide...
78,DSC 500,Teaching Assistantship (2 or 4),Training in teaching methods in the field of d...
